# Прогноз объёма сети по режиму утренней нагрузки
Модель оценивает объём следующих восьми часовых целей. Входы включают сетевую кривую за четыре часа до старта и нагрузку по группам маршрутов, которые зафиксированы в начале истории. Цель — относительное отклонение от причинного недельного профиля. Ridge и CatBoost обучаются на 17 утренних и дневных стартах. Валидация использует только 10:30. Настройки фиксируются до локального test.


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import joblib
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import pandas as pd
import pyarrow.parquet as pq
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = Path(os.environ.get('BLEND_PACKAGE', '/root/experiments/session6_blend_server'))
OOF_PATH = None
TEST_PATH = None
OUT = ROOT / 'output_regime_volume'
OUT.mkdir(parents=True, exist_ok=True)

BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
GROUP_KEYS = ['cutoff', 'horizon']
KEYS = GROUP_KEYS + ['route_id']
COLUMNS = KEYS + BASES + ['y', 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
THREADS = 2
SEED = 42

def read_panel(path, name, start, end):
    archived = ROOT / 'blend_inputs' / 'blend_best_output' / name
    if path is None and archived.is_file():
        path = archived
    if path is None:
        candidates = [p for p in ROOT.rglob(name) if OUT not in p.parents
                      and set(COLUMNS).issubset(pq.ParquetFile(p).schema.names)]
        if len(candidates) != 1:
            raise ValueError(f'Укажите путь явно для {name}: {candidates}')
        path = candidates[0]
    frame = pd.read_parquet(path, columns=COLUMNS)
    frame['cutoff'] = pd.to_datetime(frame.cutoff)
    expected = pd.date_range(start, end, inclusive='left') + pd.Timedelta(hours=10, minutes=30)
    actual = pd.DatetimeIndex(frame.cutoff.unique()).sort_values()
    counts = frame.groupby(GROUP_KEYS).size()
    if (not actual.equals(expected) or frame.duplicated(KEYS).any()
            or set(frame.horizon.unique()) != set(range(1, 9))
            or len(counts) != len(expected)*8 or not counts.eq(1000).all()
            or not np.isfinite(frame[COLUMNS[3:]].to_numpy()).all()
            or frame.y.lt(0).any() or frame.session6.lt(0).any()):
        raise ValueError(f'Некорректная панель: {path}')
    print(path, len(frame))
    return frame.reset_index(drop=True)

DATA = ROOT/'blend_inputs/session6'
BASE_RUN = Path(os.environ.get('BLEND_BASE_RUN',str(ROOT/'output/blend_submit_948a52023607')))
PARENT = Path(os.environ.get('OVERLAP_H1_PARENT',str(ROOT/'output_overlap_composite/submission_overlap_composite_full_history.csv')))
RUN_TEST = BASE_RUN/'train_val/test_predictions.parquet'
RUN_FUTURE = BASE_RUN/'full_history/future_predictions.parquet'
STEP = pd.Timedelta(minutes=30)
MIN_TEST_GAIN = .00015


In [ ]:
def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-5):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (.2, .05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-9:
                        best = loss
                        choice = source, destination, amount, trial
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(
        log_level=np.log1p(level), current=frame.current.to_numpy() / level,
        delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level,
        log_network=np.log1p(frame.network_current.to_numpy()),
        horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(),
        shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return context.clip(-50, 50).to_numpy(dtype=np.float32), level

def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

def fit_meta(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix, frame.y)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    model = CatBoostRegressor(depth=4, iterations=220, loss_function='MAE', learning_rate=.035,
                              l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False,
                              task_type='CPU', thread_count=THREADS)
    model.fit(features, (frame.y.to_numpy(dtype=float)-baseline)/level, sample_weight=level/level.mean())
    return model, weights

def predict_meta(frame, fitted):
    model, weights = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    return with_volume(frame, baseline + np.clip(model.predict(features), -.5, .5)*level)

def score_mix(frame, shape, weight):
    prediction = with_volume(frame, (1-weight)*frame.session6.to_numpy() + weight*shape)
    daily = pd.DataFrame([dict(day=str(day.date()), **metric(group.y, prediction[group.index]))
                          for day, group in frame.groupby('cutoff')])
    summary = dict(**metric(frame.y, prediction), daily_mean=daily.score.mean(),
                   daily_p90=daily.score.quantile(.9), daily_worst=daily.score.max())
    return summary, daily, prediction


In [ ]:
def load_target(path):
    raw = pd.read_parquet(path,columns=['timestamp','route_id','target_1h'])
    panel = raw.pivot(index='timestamp',columns='route_id',values='target_1h').sort_index().sort_index(axis=1)
    if (panel.shape[1]!=1000 or not panel.index.equals(pd.date_range(panel.index[0],panel.index[-1],freq=STEP))
            or not np.isfinite(panel.to_numpy()).all() or panel.lt(0).any().any()):
        raise ValueError('Некорректная история цели')
    return panel

def overlap_features(frame,level):
    x = np.zeros((len(frame),5),dtype=np.float32)
    mask = frame.horizon.eq(1).to_numpy()
    if not mask.any():
        return x
    selected = frame.loc[mask]
    positions = TARGET_PANEL.index.get_indexer(pd.DatetimeIndex(selected.cutoff))
    routes = TARGET_PANEL.columns.get_indexer(selected.route_id)
    if np.any(positions<7) or np.any(routes<0):
        raise ValueError('Не хватает причинной истории h1')
    seq = TARGET_PANEL.to_numpy(dtype=np.float64,copy=False)[positions[:,None]-np.arange(7,-1,-1),routes[:,None]]
    low = np.zeros(len(seq))
    high = seq[:,0].copy()
    valid = np.ones(len(seq),dtype=bool)
    for value in seq[:,1:].T:
        low,high = np.maximum(0,value-high),np.minimum(value,value-low)
        valid &= low<=high+1e-6
    low = np.where(valid,low,0.)
    high = np.where(valid,np.maximum(high,low),seq[:,-1])
    midpoint = (high+low)/2
    x[mask] = np.column_stack([low,high-low,midpoint,
                               midpoint-np.divide(seq[:,-1],2),valid*level[mask]])/level[mask,None]
    return x

def design(frame,matrix,baseline,augmented):
    context,level = meta_features(frame,matrix,baseline)
    if augmented:
        context = np.column_stack([context,overlap_features(frame,level)])
    return np.clip(context,-50,50).astype(np.float32,copy=False),level

def train_pair(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix,frame.y)
    baseline = matrix @ weights
    models = {}
    for name,augmented in [('control',False),('overlap_h1',True)]:
        features,level = design(frame,matrix,baseline,augmented)
        model = CatBoostRegressor(depth=4,iterations=220,loss_function='MAE',learning_rate=.035,
                                  l2_leaf_reg=20,random_seed=SEED,verbose=False,allow_writing_files=False,
                                  task_type='CPU',thread_count=4)
        model.fit(features,(frame.y.to_numpy(dtype=float)-baseline)/level,sample_weight=level/level.mean())
        models[name] = model
    return weights,models

def predict_pair(frame,fitted):
    weights,models = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    result = {}
    for name,model in models.items():
        features,level = design(frame,matrix,baseline,name=='overlap_h1')
        residual = np.clip(model.predict(features),-.5,.5)*level
        result[name] = with_volume(frame,baseline+residual)
    if 'control' in result and 'overlap_h1' in result:
        result['overlap_h1'] = np.where(frame.horizon.eq(1),result['overlap_h1'],result['control'])
    return result

def summarize(frame,forecasts,fold,period):
    rows = []
    h1 = frame.horizon.eq(1).to_numpy()
    for name,prediction in forecasts.items():
        first = metric(frame.y.to_numpy()[h1],prediction[h1])
        daily = [metric(g.y,prediction[g.index])['score'] for _,g in frame.groupby('cutoff')]
        rows.append(dict(fold=fold,period=period,variant=name,**metric(frame.y,prediction),
                         h1_WAPE=first['WAPE'],h1_score=first['score'],
                         daily_mean=float(np.mean(daily)),daily_p90=float(np.quantile(daily,.9)),
                         daily_worst=float(np.max(daily))))
    return rows


In [ ]:

SIGNALS = ['target_1h',*[f'status_{i}' for i in range(1,7)]]
METHODS = ('ridge','boost')
MIXES = (.25,.5,1.)
LIMIT = .035

def read_network(path):
    raw = pd.read_parquet(path,columns=['timestamp','route_id',*SIGNALS])
    first = raw.timestamp.min()+pd.Timedelta(days=21)
    level = raw.loc[raw.timestamp.lt(first)].groupby('route_id').target_1h.mean()
    group = pd.qcut(level.rank(method='first'),8,labels=False)
    raw['group'] = raw.route_id.map(group)
    grouped = raw.groupby(['timestamp','group'])[SIGNALS].sum().unstack('group').sort_index()
    if not grouped.index.equals(pd.date_range(grouped.index[0],grouped.index[-1],freq=STEP)):
        raise ValueError('Неполная временная сетка')
    if grouped.isna().any().any():
        raise ValueError('Неполные группы маршрутов')
    return grouped.index,grouped.to_numpy(dtype=float).reshape(len(grouped),len(SIGNALS),8)

def inputs(positions):
    if np.any(positions<48*35+8):
        raise ValueError('Не хватает истории на старте')
    seasonal_days = positions[:,None]-48*np.array([7,14,21,28,35])[None,:]
    slots = np.arange(1,9)
    seasonal = np.log(totals[seasonal_days[:,:,None]+slots[None,None,:],0])
    prior = np.exp(np.mean(seasonal,axis=1))
    recent_days = positions[:,None]-48*np.array([1,2,3,4])[None,:]
    reference = np.concatenate([seasonal_days,recent_days],axis=1)
    historical = np.log(totals[reference[:,:,None]-np.array([0,1,2,4,6,8])[None,None,:]])
    current = np.log(totals[positions[:,None]-np.array([0,1,2,4,6,8])[None,:]])
    expected = np.median(historical[:,:5],axis=1)
    deviations = current-expected
    trend = current[:,0]-current[:,-1]-(expected[:,0]-expected[:,-1])
    group_now = np.log1p(groups[positions][:,[0,2,5,6],:])
    group_prev = np.log1p(groups[reference[:,:5]][:,:,[0,2,5,6],:])
    group_deviation = group_now-np.median(group_prev,axis=1)
    group_deviation -= group_deviation.mean(axis=2,keepdims=True)
    minutes = calendar.hour.to_numpy()[positions]*60+calendar.minute.to_numpy()[positions]
    weekday = calendar.day_of_week.to_numpy()[positions]
    context = np.column_stack([deviations[:,0],deviations.mean(axis=1),trend,
                               (current-np.median(historical[:,5:],axis=1)).mean(axis=1),
                               group_deviation.reshape(len(positions),-1), minutes/60,weekday>=5])
    x = np.concatenate([np.broadcast_to(context[:,None,:],(len(positions),8,context.shape[1])),
                        np.log(prior)[...,None], seasonal.std(axis=1)[...,None],
                        np.broadcast_to(slots[None,:,None],(len(positions),8,1))],axis=2)
    return x.reshape(-1,x.shape[2]),prior

def fit_volume(stop,methods):
    eligible = TRAIN_ORIGINS[calendar[TRAIN_ORIGINS+8]<stop]
    features,prior = inputs(eligible)
    y = totals[eligible[:,None]+np.arange(1,9)[None,:],0]
    correction = np.clip(np.log(y/prior),-.3,.3).reshape(-1)
    model_set = {}
    for method in methods:
        if method=='ridge':
            model = make_pipeline(StandardScaler(),Ridge(alpha=1600))
        else:
            model = CatBoostRegressor(depth=3,iterations=280,loss_function='MAE',learning_rate=.035,
                                      l2_leaf_reg=25,random_seed=42,thread_count=4,
                                      verbose=False,allow_writing_files=False)
        model.fit(features,correction)
        model_set[method] = model
    print('Размеченные старты до',stop,len(eligible),flush=True)
    return model_set

def forecast_factors(frame,models):
    base = frame.groupby(GROUP_KEYS).session6.sum()
    cutoffs = pd.DatetimeIndex(base.index.get_level_values('cutoff').unique())
    x,prior = inputs(calendar.get_indexer(cutoffs))
    if len(base)!=len(prior)*8 or (base<=0).any():
        raise ValueError('Не совпали объёмы горизонтов')
    factors = {}
    for method,model in models.items():
        forecast = (prior*np.exp(model.predict(x).reshape(-1,8))).reshape(-1)
        ratio = forecast/base.to_numpy()-1
        for mode in ('level','curve'):
            if mode=='curve':
                shape = forecast.reshape(-1,8)
                anchor = base.to_numpy().reshape(-1,8)
                residual = (shape*(anchor.sum(axis=1)/shape.sum(axis=1))[:,None]).reshape(-1)/base.to_numpy()-1
            else:
                residual = ratio
            for mix in MIXES:
                factors[f'{method}_{mode}@{mix}'] = pd.Series(1+np.clip(residual*mix,-LIMIT,LIMIT),index=base.index)
    return factors

def predict_volume(frame,control,models):
    reference = predict_pair(frame,control)['overlap_h1']
    keys = pd.MultiIndex.from_frame(frame[GROUP_KEYS])
    forecasts = {'control':reference}
    for name,factor in forecast_factors(frame,models).items():
        forecasts[name] = reference*factor.reindex(keys).to_numpy()
    return forecasts


In [ ]:

TARGET_PANEL = load_target(DATA/'train_solo_track.parquet')
calendar,groups = read_network(DATA/'train_solo_track.parquet')
totals = groups.sum(axis=2)
minutes = calendar.hour.to_numpy()*60+calendar.minute.to_numpy()
positions = np.arange(len(calendar))
TRAIN_ORIGINS = np.flatnonzero((minutes>=390)&(minutes<=870)&(positions>=48*35+8)&(positions+8<len(calendar)))
oof = read_panel(OOF_PATH,'base_oof.parquet','2025-09-20','2025-10-18')
rows = []
for start,end in [('2025-09-27','2025-10-04'),('2025-10-04','2025-10-11'),('2025-10-11','2025-10-18')]:
    train = oof.loc[oof.cutoff+oof.horizon*STEP < pd.Timestamp(start)].reset_index(drop=True)
    val = oof.loc[oof.cutoff.between(start,end,inclusive='left')].reset_index(drop=True)
    control = train_pair(train)
    models = fit_volume(pd.Timestamp(start),METHODS)
    for period,frame in [('train_oof',train),('val',val)]:
        rows.extend(summarize(frame,predict_volume(frame,control,models),start,period))
validation = pd.DataFrame(rows)
validation.to_csv(OUT/'validation_metrics.csv',index=False)
ranking = validation.loc[validation.period.eq('val')].pivot(index='fold',columns='variant',values='daily_mean')
gains = -ranking.sub(ranking.control,axis=0)
gains.to_csv(OUT/'validation_gains.csv')
eligible = (gains.gt(0).sum()>=2)&gains.min().ge(-.00075)&gains.mean().gt(.00035)
selected = gains.loc[:,eligible].mean().sort_values(ascending=False)
winner = selected.index[0] if len(selected) else None
(OUT/'frozen_recipe.json').write_text(json.dumps(dict(winner=winner,test_used_for_tuning=False),indent=2))
display(validation)
display(gains)
print('Зафиксированный вариант',winner,flush=True)


In [ ]:

test = read_panel(RUN_TEST,'test_predictions.parquet','2025-10-18','2025-11-01')
control = train_pair(oof)
methods = [winner.split('_')[0]] if winner else []
models = fit_volume(pd.Timestamp('2025-10-18'),methods)
for period,frame in [('train_oof',oof),('test',test)]:
    forecasts = predict_volume(frame,control,models)
    keep = {name:forecasts[name] for name in ('control',winner) if name is not None}
    rows.extend(summarize(frame,keep,'2025-10-18',period))
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT/'metrics.csv',index=False)
display(metrics.loc[metrics.fold.eq('2025-10-18')])
result = metrics.loc[metrics.period.eq('test')].set_index('variant')
gain = float(result.loc['control','daily_mean']-result.loc[winner,'daily_mean']) if winner else None
passed = bool(winner and gain>=.0005 and result.loc[winner,'score']<=result.loc['control','score']+.0002)
(OUT/'decision.json').write_text(json.dumps(dict(winner=winner,passed=passed,test_daily_gain=gain,test_before_full_refit=True),indent=2))
print('Решение',passed,'выигрыш',gain,flush=True)
if passed:
    future = pd.read_parquet(RUN_FUTURE,columns=KEYS+['session6'])
    future['cutoff'] = pd.to_datetime(future.cutoff)
    request = pd.read_parquet(DATA/'test_solo_track.parquet',columns=['id','route_id','timestamp'])
    parent = pd.read_csv(PARENT)
    cutoff = request.timestamp.min()-STEP
    if (len(request)!=8000 or len(parent)!=8000 or set(request.id)!=set(parent.id)
            or not request.id.is_unique or not parent.id.is_unique or not future.cutoff.eq(cutoff).all()
            or calendar[-1]!=cutoff):
        raise ValueError('Несовместимые данные для сабмита')
    final_models = fit_volume(cutoff+STEP,methods)
    factor = forecast_factors(future,final_models)[winner]
    query = request.merge(parent,on='id',validate='one_to_one',sort=False)
    query['cutoff'] = cutoff
    query['horizon'] = ((query.timestamp-cutoff)/STEP).astype(int)
    prediction = query.y_pred.to_numpy()*factor.reindex(pd.MultiIndex.from_frame(query[GROUP_KEYS])).to_numpy()
    if not np.isfinite(prediction).all() or (prediction<0).any():
        raise ValueError('Некорректный сабмит')
    destination = OUT/'submission_regime_volume_full_history.csv'
    query[['id']].assign(y_pred=prediction).to_csv(destination.with_suffix('.partial'),index=False)
    destination.with_suffix('.partial').replace(destination)
    factor.rename('factor').to_csv(OUT/'contest_factors.csv')
    joblib.dump(final_models,OUT/'volume_models.joblib')
    print('Сабмит',destination,flush=True)
print('Результаты',OUT,flush=True)
